In [36]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from statsmodels.genmod.generalized_linear_model import SET_USE_BIC_LLF
SET_USE_BIC_LLF(True)

import json
from xgboost import XGBClassifier
import shap

from sklearn.metrics import accuracy_score, f1_score, classification_report, make_scorer, roc_auc_score
from sklearn.model_selection import cross_validate
from statsmodels.stats.outliers_influence import variance_inflation_factor


In [2]:
with open("PROCESSED/DATA/merged_and_dropped.cat_cols.json") as f:
    cat_cols = json.load(f)

X_train = pd.read_parquet("INPUTS/TRAIN/X_train.parquet")
X_test = pd.read_parquet("INPUTS/TEST/X_test.parquet")
y_train = pd.read_parquet("INPUTS/TRAIN/y_train.parquet")
y_test = pd.read_parquet("INPUTS/TEST/y_test.parquet")

X_train[cat_cols] = X_train[cat_cols].astype("category")
X_test[cat_cols] = X_test[cat_cols].astype("category")

num_cols = [c for c in X_train.columns if c not in cat_cols]

# drop leakage
X_train = X_train.drop(columns=["P_BIOPRO__LBXSOSSI_Osmolality_mmol_Kg"])
X_test = X_test.drop(columns=["P_BIOPRO__LBXSOSSI_Osmolality_mmol_Kg"])

# one-hot encode categorical variables
X_train_encoded = pd.get_dummies(X_train, drop_first=True)
X_test_encoded = pd.get_dummies(X_test, drop_first=True)
X_test_encoded = X_test_encoded.reindex(columns=X_train_encoded.columns, fill_value=0)

# ensure target is categorical
y_train_cat = y_train.iloc[:, 0].astype("category")
y_test_cat  = y_test.iloc[:, 0].astype("category")

In [3]:
# best parameters
with open("RESULTS/BASELINES/PARAMETERS/XGBClassifier_BEST_HYPER.json") as f:
    xgb_param = json.load(f)

# model with best parameters
xgb = XGBClassifier(
    objective='binary:logistic',
    eval_metric='logloss',
    tree_method='hist',
    random_state=42,
    n_jobs=-1,
    **xgb_param
)

# cross-validation
scoring = {'accuracy': 'accuracy',
           'f1': make_scorer(f1_score, average='macro'),
           'auc': 'roc_auc'}
cv_results = cross_validate(xgb, X_train_encoded, y_train_cat, cv=5, scoring=scoring)

# fit
xgb.fit(X_train_encoded, y_train_cat)

# evaluate
y_pred_train = xgb.predict(X_train_encoded)
y_pred_test  = xgb.predict(X_test_encoded)

acc_train = accuracy_score(y_train_cat, y_pred_train)
acc_test  = accuracy_score(y_test_cat, y_pred_test)

f1_train = f1_score(y_train_cat, y_pred_train, average='macro')
f1_test  = f1_score(y_test_cat, y_pred_test, average='macro')

auc_train = roc_auc_score(y_train_cat, xgb.predict_proba(X_train_encoded)[:, 1])
auc_test  = roc_auc_score(y_test_cat, xgb.predict_proba(X_test_encoded)[:, 1])

# print(f"CV accuracy:    {cv_results['test_accuracy'].mean():.3f} ± {cv_results['test_accuracy'].std():.3f}")
# print(f"CV F1:          {cv_results['test_f1'].mean():.3f} ± {cv_results['test_f1'].std():.3f}")
# print(f"CV AUC:         {cv_results['test_auc'].mean():.3f} ± {cv_results['test_auc'].std():.3f}")


print(f"CV accuracy    : {cv_results['test_accuracy'].mean():.3f}, F1: {cv_results['test_f1'].mean():.3f}, AUC: {cv_results['test_auc'].mean():.3f}")
print(f"Train accuracy : {acc_train:.3f}, F1: {f1_train:.3f}, AUC: {auc_train:.3f}")
print(f"Test  accuracy : {acc_test:.3f}, F1: {f1_test:.3f}, AUC: {auc_test:.3f}")

print("\nClassification report:\n")
print(classification_report(y_test_cat, y_pred_test))

CV accuracy    : 0.873, F1: 0.744, AUC: 0.903
Train accuracy : 0.940, F1: 0.883, AUC: 0.976
Test  accuracy : 0.902, F1: 0.790, AUC: 0.921

Classification report:

              precision    recall  f1-score   support

         0.0       0.92      0.97      0.94      1642
         1.0       0.76      0.55      0.64       306

    accuracy                           0.90      1948
   macro avg       0.84      0.76      0.79      1948
weighted avg       0.89      0.90      0.90      1948



#### Base Model

In [4]:
X_train_glm_untuned = sm.add_constant(X_train_encoded.astype(float))
X_test_glm_untuned = sm.add_constant(X_test_encoded.astype(float))

glm_untuned = sm.GLM(
    y_train,
    X_train_glm_untuned,
    family = sm.families.Binomial()
).fit()

# Predictions
y_pred_train_glm_untuned = glm_untuned.predict(X_train_glm_untuned)
y_pred_test_glm_untuned = glm_untuned.predict(X_test_glm_untuned)

y_pred_train_cat_untuned = (y_pred_train_glm_untuned >= 0.5).astype(int)
y_pred_test_cat_untuned = (y_pred_test_glm_untuned >= 0.5).astype(int)
acc_train_untuned = accuracy_score(y_train_cat, y_pred_train_cat_untuned)
acc_test_untuned = accuracy_score(y_test_cat, y_pred_test_cat_untuned)
f1_train_untuned = f1_score(y_train_cat, y_pred_train_cat_untuned, average='macro')
f1_test_untuned = f1_score(y_test_cat, y_pred_test_cat_untuned, average='macro')
auc_train_untuned = roc_auc_score(y_train_cat, y_pred_train_glm_untuned)
auc_test_untuned = roc_auc_score(y_test_cat, y_pred_test_glm_untuned)

print(f"Train accuracy: {acc_train_untuned:.3f},  F1: {f1_train_untuned:.3f}, AUC: {auc_train_untuned:.3f}")
print(f"Test  accuracy: {acc_test_untuned:.3f},  F1: {f1_test_untuned:.3f}, AUC: {auc_test_untuned:.3f}")
print("\nClassification report:\n")
cls_report = classification_report(y_test_cat, y_pred_test_cat_untuned)
print(cls_report)

# Probabilty Outputs
test_df_untuned = pd.DataFrame(y_pred_test_glm_untuned, columns=["prob_1"])
test_df_untuned["y_true"] = np.asarray(y_test)

# Performance metrics
model_name = "GLM_UNTUNED"
perf_row = {
    "model": model_name,
    "acc_train": acc_train_untuned,
    "acc_test": acc_test_untuned,
    "f1_train": f1_train_untuned,
    "f1_test": f1_test_untuned,
    "auc_train": auc_train_untuned,
    "auc_test": auc_test_untuned,
}
perf_df = pd.DataFrame([perf_row])

print(glm_untuned.summary())

c:\Users\victo\AppData\Local\Programs\Python\Python312\Lib\site-packages\statsmodels\genmod\families\links.py:198: RuntimeWarning: overflow encountered in exp
  t = np.exp(-z)


Train accuracy: 0.857,  F1: 0.634, AUC: 0.606
Test  accuracy: 0.863,  F1: 0.626, AUC: 0.598

Classification report:

              precision    recall  f1-score   support

         0.0       0.87      0.98      0.92      1642
         1.0       0.71      0.21      0.33       306

    accuracy                           0.86      1948
   macro avg       0.79      0.60      0.63      1948
weighted avg       0.85      0.86      0.83      1948



c:\Users\victo\AppData\Local\Programs\Python\Python312\Lib\site-packages\statsmodels\genmod\families\links.py:198: RuntimeWarning: overflow encountered in exp
  t = np.exp(-z)
c:\Users\victo\AppData\Local\Programs\Python\Python312\Lib\site-packages\statsmodels\genmod\families\family.py:1056: RuntimeWarning: divide by zero encountered in log
  special.gammaln(n - y + 1) + y * np.log(mu / (1 - mu + 1e-20)) +
c:\Users\victo\AppData\Local\Programs\Python\Python312\Lib\site-packages\statsmodels\genmod\families\family.py:1056: RuntimeWarning: invalid value encountered in multiply
  special.gammaln(n - y + 1) + y * np.log(mu / (1 - mu + 1e-20)) +


                 Generalized Linear Model Regression Results                  
Dep. Variable:            IS_DIABETES   No. Observations:                 7789
Model:                            GLM   Df Residuals:                     7419
Model Family:                Binomial   Df Model:                          369
Link Function:                  Logit   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                    nan
Date:                Tue, 25 Nov 2025   Deviance:                   1.0223e+05
Time:                        10:43:45   Pearson chi2:                 5.00e+18
No. Iterations:                   100   Pseudo R-squ. (CS):                nan
Covariance Type:            nonrobust                                         
                                                                    coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------

#### Variable Selected by Trees Importance

In [5]:
# gain feature importance
booster = xgb.get_booster()
score = booster.get_score(importance_type='gain')

importance_df = pd.DataFrame({
    'feature': list(score.keys()),
    'importance': list(score.values())
}).sort_values('importance', ascending=False)

top_n = 100
top_features = importance_df['feature'].head(top_n).tolist()

# fit
X_train_glm_top = X_train_encoded[top_features]
X_test_glm_top = X_test_encoded[top_features]

X_train_glm_top = sm.add_constant(X_train_glm_top.astype(float))
X_test_glm_top = sm.add_constant(X_test_glm_top.astype(float))

glm_top = sm.GLM(
    y_train,
    X_train_glm_top,
    family=sm.families.Binomial()
).fit()


# Predictions
y_pred_train_glm_top = glm_top.predict(X_train_glm_top)
y_pred_test_glm_top = glm_top.predict(X_test_glm_top)

y_pred_train_cat_top = (y_pred_train_glm_top >= 0.5).astype(int)
y_pred_test_cat_top = (y_pred_test_glm_top >= 0.5).astype(int)
acc_train_top = accuracy_score(y_train_cat, y_pred_train_cat_top)
acc_test_top = accuracy_score(y_test_cat, y_pred_test_cat_top)
f1_train_top = f1_score(y_train_cat, y_pred_train_cat_top, average='macro')
f1_test_top = f1_score(y_test_cat, y_pred_test_cat_top, average='macro')
auc_train_top = roc_auc_score(y_train_cat, y_pred_train_glm_top)
auc_test_top = roc_auc_score(y_test_cat, y_pred_test_glm_top)
print(f"Train accuracy: {acc_train_top:.3f},  F1: {f1_train_top:.3f}, AUC: {auc_train_top:.3f}")
print(f"Test  accuracy: {acc_test_top:.3f},  F1: {f1_test_top:.3f}, AUC: {auc_test_top:.3f}")
print("\nClassification report:\n")
cls_report = classification_report(y_test_cat, y_pred_test_cat_top)
print(cls_report)

# Probabilty Outputs
test_df_top = pd.DataFrame(y_pred_test_glm_top, columns=["prob_1"])
test_df_top["y_true"] = np.asarray(y_test)

# Performance metrics
model_name = "GLM_TOP_FEATURES"
perf_row = {
    "model": model_name,
    "acc_train": acc_train_top,
    "acc_test": acc_test_top,
    "f1_train": f1_train_top,
    "f1_test": f1_test_top,
    "auc_train": auc_train_top,
    "auc_test": auc_test_top,
}
perf_df = pd.DataFrame([perf_row])

print(glm_top.summary())

Train accuracy: 0.882,  F1: 0.764, AUC: 0.906
Test  accuracy: 0.887,  F1: 0.761, AUC: 0.899

Classification report:

              precision    recall  f1-score   support

         0.0       0.91      0.96      0.93      1642
         1.0       0.69      0.51      0.59       306

    accuracy                           0.89      1948
   macro avg       0.80      0.73      0.76      1948
weighted avg       0.88      0.89      0.88      1948

                 Generalized Linear Model Regression Results                  
Dep. Variable:            IS_DIABETES   No. Observations:                 7789
Model:                            GLM   Df Residuals:                     7688
Model Family:                Binomial   Df Model:                          100
Link Function:                  Logit   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -2146.5
Date:                Tue, 25 Nov 2025   Deviance:                       4293.1
Tim

#### Variable Selected by SHAP Importance

In [6]:
explainer = shap.TreeExplainer(xgb)
shap_values = explainer.shap_values(X_train_encoded)
shap_vals = shap_values.values if hasattr(shap_values, "values") else shap_values
shap_feature_importance = np.abs(shap_vals).mean(axis=0)
shap_importance_df = pd.DataFrame({
    'feature': X_train_encoded.columns,
    'importance': shap_feature_importance
}).sort_values('importance', ascending=False)

top_n_shap = 100
top_features_shap = shap_importance_df['feature'].head(top_n_shap).tolist()

# fit
X_train_glm_shap = X_train_encoded[top_features_shap]
X_test_glm_shap = X_test_encoded[top_features_shap]

X_train_glm_shap = sm.add_constant(X_train_glm_shap.astype(float))
X_test_glm_shap = sm.add_constant(X_test_glm_shap.astype(float))

glm_shap = sm.GLM(
    y_train,
    X_train_glm_shap,
    family=sm.families.Binomial()
).fit()


# Predictions
y_pred_train_glm_shap = glm_shap.predict(X_train_glm_shap)
y_pred_test_glm_shap = glm_shap.predict(X_test_glm_shap)

y_pred_train_cat_shap = (y_pred_train_glm_shap >= 0.5).astype(int)
y_pred_test_cat_shap = (y_pred_test_glm_shap >= 0.5).astype(int)
acc_train_shap = accuracy_score(y_train_cat, y_pred_train_cat_shap)
acc_test_shap = accuracy_score(y_test_cat, y_pred_test_cat_shap)
f1_train_shap = f1_score(y_train_cat, y_pred_train_cat_shap, average='macro')
f1_test_shap = f1_score(y_test_cat, y_pred_test_cat_shap, average='macro')
auc_train_shap = roc_auc_score(y_train_cat, y_pred_train_glm_shap)
auc_test_shap = roc_auc_score(y_test_cat, y_pred_test_glm_shap)
print(f"Train accuracy: {acc_train_shap:.3f},  F1: {f1_train_shap:.3f}, AUC: {auc_train_shap:.3f}")
print(f"Test  accuracy: {acc_test_shap:.3f},  F1: {f1_test_shap:.3f}, AUC: {auc_test_shap:.3f}")
print("\nClassification report:\n")
cls_report = classification_report(y_test_cat, y_pred_test_cat_shap)
print(cls_report)

# Probabilty Outputs
test_df_shap = pd.DataFrame(y_pred_test_glm_shap, columns=["prob_1"])
test_df_shap["y_true"] = np.asarray(y_test)

# Performance metrics
model_name = "GLM_TOP_FEATURES"
perf_row = {
    "model": model_name,
    "acc_train": acc_train_shap,
    "acc_test": acc_test_shap,
    "f1_train": f1_train_shap,
    "f1_test": f1_test_shap,
    "auc_train": auc_train_shap,
    "auc_test": auc_test_shap,
}
perf_df = pd.DataFrame([perf_row])

print(glm_shap.summary())

Train accuracy: 0.881,  F1: 0.764, AUC: 0.909
Test  accuracy: 0.888,  F1: 0.765, AUC: 0.903

Classification report:

              precision    recall  f1-score   support

         0.0       0.92      0.95      0.93      1642
         1.0       0.69      0.53      0.60       306

    accuracy                           0.89      1948
   macro avg       0.80      0.74      0.76      1948
weighted avg       0.88      0.89      0.88      1948

                 Generalized Linear Model Regression Results                  
Dep. Variable:            IS_DIABETES   No. Observations:                 7789
Model:                            GLM   Df Residuals:                     7688
Model Family:                Binomial   Df Model:                          100
Link Function:                  Logit   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -2117.3
Date:                Tue, 25 Nov 2025   Deviance:                       4234.6
Tim

### Statistical Tuning

### Significance Backward Elimination

In [7]:
def backward_elimination_classification(X_train, y_train, p_threshold=0.05):
    """
    Backward elimination for classification (Binomial GLM).
    Removes highest p-value feature iteratively.
    """
    X_current = X_train.copy()
    removed_features = []
    iteration = 0

    while True:
        iteration += 1

        X_const = sm.add_constant(X_current.astype(float))

        model = sm.GLM(
            y_train,
            X_const,
            family=sm.families.Binomial()
        ).fit()

        pvalues = model.pvalues[1:]
        max_p = pvalues.max()
        worst_feat = pvalues.idxmax()

        y_pred_prob = model.predict(X_const)
        y_pred_class = (y_pred_prob >= 0.5).astype(int)

        acc = accuracy_score(y_train, y_pred_class)
        f1 = f1_score(y_train, y_pred_class, average='macro')
        auc = roc_auc_score(y_train, y_pred_prob)

        print("\n" + "="*80)
        print(f"Iteration {iteration}")
        print("="*80)
        print(f"Features remaining: {len(X_current.columns)}")
        print(f"Worst feature: {worst_feat}")
        print(f"Worst p-value: {max_p:.4f}")
        print(f"Train ACC: {acc:.4f}, F1: {f1:.4f}, AUC: {auc:.4f}")

        if max_p < p_threshold:
            print(f"\n✓ All features significant (p < {p_threshold})")
            break

        print(f"→ Removing: {worst_feat}")
        removed_features.append((worst_feat, max_p))
        X_current = X_current.drop(columns=[worst_feat])

    print("\n" + "="*80)
    print("BACKWARD ELIMINATION COMPLETE")
    print("="*80)
    print(f"Features removed: {len(removed_features)}")
    print(f"Features remaining: {len(X_current.columns)}")
    print("\nRemoved features:")
    for feat, pval in removed_features:
        print(f"  - {feat} (p = {pval:.4f})")

    return X_current, model, removed_features


# apply backward elimination
X_train_be, be_model, removed = backward_elimination_classification(
    X_train_glm_shap, y_train_cat, p_threshold=0.05
)

X_test_be = X_test_glm_shap[X_train_be.columns]

X_train_be_const = sm.add_constant(X_train_be)
X_test_be_const = sm.add_constant(X_test_be)

y_train_prob = be_model.predict(X_train_be_const)
y_test_prob = be_model.predict(X_test_be_const)

y_train_pred = (y_train_prob >= 0.5).astype(int)
y_test_pred = (y_test_prob >= 0.5).astype(int)

acc_train = accuracy_score(y_train_cat, y_train_pred)
acc_test = accuracy_score(y_test_cat, y_test_pred)
f1_train = f1_score(y_train_cat, y_train_pred, average='macro')
f1_test = f1_score(y_test_cat, y_test_pred, average='macro')
auc_train = roc_auc_score(y_train_cat, y_train_prob)
auc_test = roc_auc_score(y_test_cat, y_test_prob)

print("\nFINAL RESULTS")
print(f"Train ACC: {acc_train:.3f}, F1: {f1_train:.3f}, AUC: {auc_train:.3f}")
print(f"Test  ACC: {acc_test:.3f}, F1: {f1_test:.3f}, AUC: {auc_test:.3f}")



Iteration 1
Features remaining: 101
Worst feature: P_IHGEM__LBDBGMSI_Mercury_methyl_nmol_L
Worst p-value: 0.9490
Train ACC: 0.8811, F1: 0.7643, AUC: 0.9091
→ Removing: P_IHGEM__LBDBGMSI_Mercury_methyl_nmol_L

Iteration 2
Features remaining: 100
Worst feature: P_ALB_CR__URXUMS_Albumin_urine_mg_L
Worst p-value: 0.9429
Train ACC: 0.8809, F1: 0.7638, AUC: 0.9091
→ Removing: P_ALB_CR__URXUMS_Albumin_urine_mg_L

Iteration 3
Features remaining: 99
Worst feature: P_FASTQX__PHAFSTMN_Total_length_of_food_fast_minutes
Worst p-value: 0.9349
Train ACC: 0.8810, F1: 0.7640, AUC: 0.9091
→ Removing: P_FASTQX__PHAFSTMN_Total_length_of_food_fast_minutes

Iteration 4
Features remaining: 98
Worst feature: P_BIOPRO__LBDSTPSI_Total_Protein_g_L
Worst p-value: 0.9345
Train ACC: 0.8810, F1: 0.7640, AUC: 0.9091
→ Removing: P_BIOPRO__LBDSTPSI_Total_Protein_g_L

Iteration 5
Features remaining: 97
Worst feature: P_BIOPRO__LBXSCK_Creatine_Phosphokinase_CPK_IU_L
Worst p-value: 0.8943
Train ACC: 0.8811, F1: 0.7643, A

#### VIF Drop

In [8]:
# apply VIF with thresholds
numeric_cols = X_train_be.select_dtypes(include=['number']).columns.tolist()
X_vif = X_train_be[numeric_cols].dropna()

vif_thresholds = [40, 30, 20, 10, 5]
threshold_results = {}
current_threshold_idx = 0

# Initialize
vif_summary = pd.DataFrame({'feature': X_vif.columns})

iteration = 0
print(f"Starting with {len(X_vif.columns)} features\n")

while current_threshold_idx < len(vif_thresholds):
    iteration += 1
    
    # Calculate VIF
    vif_values = [variance_inflation_factor(X_vif.values, i) for i in range(X_vif.shape[1])]
    max_vif = max(vif_values)
    
    vif = pd.DataFrame({
        'feature': X_vif.columns,
        f'VIF_iter{iteration}': vif_values
    })
    
    vif_summary = vif_summary.merge(vif, on='feature', how='left')
    
    current_threshold = vif_thresholds[current_threshold_idx]
    
    # Check if we've dropped below current threshold
    if max_vif < current_threshold:
        print(f"\n{'='*60}")
        print(f"VIF < {current_threshold} REACHED")
        print(f"{'='*60}")
        
        # Save results for this threshold
        final_features = X_vif.columns.tolist()
        threshold_results[current_threshold] = {
            'features': final_features,
            'n_features': len(final_features)
        }
        
        # Save files - ALL AS CSV IN INPUT/MISC
        # vif_summary.to_csv(f"LOG/VIF_log_threshold_{current_threshold}.csv", index=False)
        # pd.DataFrame({'feature': final_features}).to_csv(
        #     f"INPUT/MISC/selected_features_vif_{current_threshold}.csv", index=False
        # )
        
        print(f"Features remaining: {len(final_features)}")
        print(f"Features removed: {len(numeric_cols) - len(final_features)}")
        
        # Move to next threshold
        current_threshold_idx += 1
        
        # If we've reached the last threshold, break
        if current_threshold_idx >= len(vif_thresholds):
            break
        
        continue
    
    # Find max VIF and drop
    max_idx = vif_values.index(max_vif)
    drop_col = X_vif.columns[max_idx]
    
    print(f"[{iteration}] Drop {drop_col}: VIF={max_vif:.1f}")
    X_vif = X_vif.drop(columns=drop_col)

# Summary comparison
print("\n" + "="*60)
print("SUMMARY: Feature Count by VIF Threshold")
print("="*60)
print(f"{'Threshold':<12} {'Features':<12} {'Removed':<12} {'% Remaining':<12}")
print("-"*60)

for threshold in [40, 30, 20, 10, 5]:
    if threshold in threshold_results:
        n_features = threshold_results[threshold]['n_features']
        n_removed = len(numeric_cols) - n_features
        pct_remaining = (n_features / len(numeric_cols)) * 100
        print(f"VIF < {threshold:<6} {n_features:<12} {n_removed:<12} {pct_remaining:.1f}%")

# Save comparison summary
comparison_df = pd.DataFrame({
    'VIF_Threshold': list(threshold_results.keys()),
    'Features_Remaining': [threshold_results[t]['n_features'] for t in threshold_results.keys()],
    'Features_Removed': [len(numeric_cols) - threshold_results[t]['n_features'] for t in threshold_results.keys()],
    'Percent_Remaining': [(threshold_results[t]['n_features'] / len(numeric_cols)) * 100 for t in threshold_results.keys()]
})
# comparison_df.to_csv("LOG/VIF_threshold_comparison.csv", index=False)

# print(f"\nFiles saved:")
# print(f"  - LOG/VIF_log_threshold_[5|10|20|30|40].csv")
# print(f"  - INPUT/MISC/selected_features_vif_[5|10|20|30|40].csv")
# print(f"  - LOG/VIF_threshold_comparison.csv")

Starting with 57 features

[1] Drop const: VIF=4945.2
[2] Drop P_BIOPRO__LBXSNASI_Sodium_mmol_L: VIF=6507.2
[3] Drop P_BIOPRO__LBDSCASI_Total_Calcium_mmol_L: VIF=843.3
[4] Drop P_BIOPRO__LBXSCLSI_Chloride_mmol_L: VIF=509.7
[5] Drop P_BMX__BMXHIP_Hip_Circumference_cm: VIF=470.9
[6] Drop P_BMX__BMXWAIST_Waist_Circumference_cm: VIF=315.3
[7] Drop P_BMX__BMXLEG_Upper_Leg_Length_cm: VIF=173.9
[8] Drop P_BIOPRO__LBXSAL_Albumin_refrigerated_serum_g_dL: VIF=156.3
[9] Drop BP_sys_median: VIF=123.2
[10] Drop P_BIOPRO__LBXSC3SI_Bicarbonate_mmol_L: VIF=97.5
[11] Drop P_CBC__LBXMPSI_Mean_platelet_volume_fL: VIF=67.4
[12] Drop P_BMX__BMXBMI_Body_Mass_Index_kg_m_2: VIF=64.6
[13] Drop P_BIOPRO__LBDSGBSI_Globulin_g_L: VIF=61.2
[14] Drop BP_dia_median: VIF=55.2
[15] Drop P_PBCD__LBDBSESI_Blood_selenium_umol_L: VIF=43.9
[16] Drop Pulse_median: VIF=40.1

VIF < 40 REACHED
Features remaining: 41
Features removed: 16

VIF < 30 REACHED
Features remaining: 41
Features removed: 16
[19] Drop P_LUX__LUXCAPM_Media

In [13]:
# choose VIF cutoff
n_VIF_thresh = 20
print(f"\nEvaluating GLM with VIF < {n_VIF_thresh} features\n")

# get selected feature list
vif_features = threshold_results[n_VIF_thresh]['features']

# subset TRAIN and TEST using those features
X_train_vif = X_train_be[vif_features]
X_test_vif  = X_test_be[vif_features]

# add constant
X_train_vif_const = sm.add_constant(X_train_vif.astype(float))
X_test_vif_const  = sm.add_constant(X_test_vif.astype(float))

# refit GLM
glm_vif = sm.GLM(
    y_train,
    X_train_vif_const,
    family=sm.families.Binomial()
).fit()

# predictions
y_train_prob_vif = glm_vif.predict(X_train_vif_const)
y_test_prob_vif  = glm_vif.predict(X_test_vif_const)

y_train_pred_vif = (y_train_prob_vif >= 0.5).astype(int)
y_test_pred_vif  = (y_test_prob_vif >= 0.5).astype(int)

# metrics
acc_train_vif = accuracy_score(y_train_cat, y_train_pred_vif)
acc_test_vif  = accuracy_score(y_test_cat, y_test_pred_vif)
f1_train_vif  = f1_score(y_train_cat, y_train_pred_vif, average='macro')
f1_test_vif   = f1_score(y_test_cat, y_test_pred_vif, average='macro')
auc_train_vif = roc_auc_score(y_train_cat, y_train_prob_vif)
auc_test_vif  = roc_auc_score(y_test_cat, y_test_prob_vif)

print(f"Train ACC: {acc_train_vif:.3f}, F1: {f1_train_vif:.3f}, AUC: {auc_train_vif:.3f}")
print(f"Test  ACC: {acc_test_vif:.3f}, F1: {f1_test_vif:.3f}, AUC: {auc_test_vif:.3f}\n")

print(glm_vif.summary())


Evaluating GLM with VIF < 20 features

Train ACC: 0.861, F1: 0.708, AUC: 0.874
Test  ACC: 0.864, F1: 0.699, AUC: 0.871

                 Generalized Linear Model Regression Results                  
Dep. Variable:            IS_DIABETES   No. Observations:                 7789
Model:                            GLM   Df Residuals:                     7751
Model Family:                Binomial   Df Model:                           37
Link Function:                  Logit   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -2438.2
Date:                Tue, 25 Nov 2025   Deviance:                       4876.5
Time:                        10:46:05   Pearson chi2:                 8.02e+03
No. Iterations:                     7   Pseudo R-squ. (CS):             0.2539
Covariance Type:            nonrobust                                         
                                                                coef    std err          

In [ ]:
# def stepwise_glm_aicbic(X, y, criterion="AIC"):
#     """
#     Backward elimination using AIC/BIC for Binomial GLM
#     criterion: "AIC" or "BIC"
#     """
#     X_current = X.copy()
#     history = []

#     while True:
#         best_score = None
#         worst_feature = None
#         current_const = sm.add_constant(X_current.astype(float))
#         base_model = sm.GLM(y, current_const, family=sm.families.Binomial()).fit()

#         if criterion.upper() == "AIC":
#             base_score = base_model.aic
#         else:
#             base_score = base_model.bic

#         for feature in X_current.columns:
#             X_temp = X_current.drop(columns=[feature])
#             X_temp_const = sm.add_constant(X_temp.astype(float))
#             model_temp = sm.GLM(y, X_temp_const, family=sm.families.Binomial()).fit()

#             score_temp = model_temp.aic if criterion.upper() == "AIC" else model_temp.bic

#             if best_score is None or score_temp < best_score:
#                 best_score = score_temp
#                 worst_feature = feature

#         history.append((len(X_current.columns), base_score))

#         if best_score >= base_score:
#             break

#         X_current = X_current.drop(columns=[worst_feature])

#     final_const = sm.add_constant(X_current.astype(float))
#     final_model = sm.GLM(y, final_const, family=sm.families.Binomial()).fit()

#     return X_current, final_model, history


# X_step_aic, model_aic, history_aic = stepwise_glm_aicbic(X_train_be, y_train, criterion="AIC")

In [38]:
def stepwise_glm_aic_backward(X, y, criterion="AIC", verbose=True):

    crit = criterion.upper()
    X_current = X.copy()
    history = []
    iteration = 0

    while True:
        iteration += 1

        X_const = sm.add_constant(X_current.astype(float))
        base_model = sm.GLM(y, X_const, family=sm.families.Binomial()).fit()

        base_score = base_model.aic if crit=="AIC" else base_model.bic
        base_rss = base_model.deviance

        if verbose:
            print(f"\nStep:  {crit}={base_score:.3f}")
            if len(X_current.columns) > 0:
                print("y ~ " + " + ".join(X_current.columns))
            else:
                print("y ~ 1")
            print()
            print(f"{'Df':<6}{'Sum of Sq':<14}{'RSS':<12}{crit:<8}")
            print("-"*50)

        best_score = base_score
        worst_feature = None
        best_model = base_model

        # <none> row
        if verbose:
            print(f"{'<none>':<6}{'-':<14}{base_rss:<12.3f}{base_score:<8.3f}")

        for feature in X_current.columns:
            X_temp = X_current.drop(columns=[feature])
            X_temp_const = sm.add_constant(X_temp.astype(float))
            model_temp = sm.GLM(y, X_temp_const, family=sm.families.Binomial()).fit()

            rss = model_temp.deviance
            sumsq = rss - base_rss
            score = model_temp.aic if crit=="AIC" else model_temp.bic

            if verbose:
                print(f"- {feature:<4}{1:<6}{sumsq:<14.3f}{rss:<12.3f}{score:<8.3f}")

            if score < best_score:
                best_score = score
                worst_feature = feature
                best_model = model_temp

        if worst_feature is None:
            if verbose:
                print("\nNo improvement — stopping")
            break

        if verbose:
            print(f"\nSelected: - {worst_feature}  ({crit}={best_score:.3f})")

        history.append({
            "Step": iteration,
            "Removed": worst_feature,
            crit: best_score
        })

        X_current = X_current.drop(columns=[worst_feature])

    return X_current, best_model, history



X_step_aic_backward, model_aic_backward, history_aic_backward = stepwise_glm_aic_backward(X_train_be, y_train, criterion="AIC")


Step:  AIC=4398.318
y ~ const + P_DEMO__RIDAGEYR_Age_in_years_at_screening + P_MCQ__MCQ300C + P_ALB_CR__URDACT_Albumin_creatinine_ratio_mg_g + P_LUX__LUXCAPM_Median_CAP_decibels_per_meter_dB_m + P_BIOPRO__LBXSCLSI_Chloride_mmol_L + Pulse_median + P_BIOPRO__LBDSTRSI_Triglycerides_refrig_serum_mmol_L + P_BMX__BMXWAIST_Waist_Circumference_cm + P_FASTQX__PHDSESN_Session_in_which_SP_was_examined_1.0 + P_TCHOL__LBDTCSI_Total_Cholesterol_mmol_L + P_BPQ__BPQ020_Ever_told_you_had_high_blood_pressure_2.0 + P_PBCD__LBDBPBSI_Blood_lead_umol_L + P_MCQ__MCQ366D + P_BIOPRO__LBXSLDSI_Lactate_Dehydrogenase_LDH_IU_L + P_DEMO__RIDRETH3_Race_Hispanic_origin_w_NH_Asian_3.0 + P_WHQ__WHQ150_Age_when_heaviest_weight + P_PAQ__PAQ620_Moderate_work_activity_2.0 + P_MCQ__MCQ366B + P_LUX__LUXSMED_Median_stiffness_E_kilopascals_kPa + P_TST__LBXSHBG_SHBG_nmol_L + P_WHQ__WHD140_Self_reported_greatest_weight_pounds + P_CBC__LBXMPSI_Mean_platelet_volume_fL + BP_dia_median + P_FASTQX__PHDSESN_Session_in_which_SP_was_ex

In [ ]:
X_train_aic_const = sm.add_constant(X_step_aic_backward.astype(float))
X_test_aic_const  = sm.add_constant(X_test_be[X_step_aic_backward.columns].astype(float))

y_train_prob_aic = model_aic_backward.predict(X_train_aic_const)
y_test_prob_aic  = model_aic_backward.predict(X_test_aic_const)

y_train_pred_aic = (y_train_prob_aic >= 0.5).astype(int)
y_test_pred_aic  = (y_test_prob_aic >= 0.5).astype(int)

acc_train_aic = accuracy_score(y_train_cat, y_train_pred_aic)
acc_test_aic  = accuracy_score(y_test_cat, y_test_pred_aic)
f1_train_aic  = f1_score(y_train_cat, y_train_pred_aic, average='macro')
f1_test_aic   = f1_score(y_test_cat, y_test_pred_aic, average='macro')
auc_train_aic = roc_auc_score(y_train_cat, y_train_prob_aic)
auc_test_aic  = roc_auc_score(y_test_cat, y_test_prob_aic)

print(f"Train ACC: {acc_train_aic:.3f}, F1: {f1_train_aic:.3f}, AUC: {auc_train_aic:.3f}")
print(f"Test  ACC: {acc_test_aic:.3f}, F1: {f1_test_aic:.3f}, AUC: {auc_test_aic:.3f}\n")

print(model_aic_backward.summary())


Train ACC: 0.878, F1: 0.758, AUC: 0.907
Test  ACC: 0.883, F1: 0.751, AUC: 0.902

                 Generalized Linear Model Regression Results                  
Dep. Variable:            IS_DIABETES   No. Observations:                 7789
Model:                            GLM   Df Residuals:                     7732
Model Family:                Binomial   Df Model:                           56
Link Function:                  Logit   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -2142.2
Date:                Tue, 25 Nov 2025   Deviance:                       4284.3
Time:                        11:26:02   Pearson chi2:                 7.00e+03
No. Iterations:                     7   Pseudo R-squ. (CS):             0.3085
Covariance Type:            nonrobust                                         
                                                                coef    std err          z      P>|z|      [0.025      0.975]
---

In [44]:
X_step_bic_backward, model_bic_backward, history_bic_backward = stepwise_glm_aic_backward(X_train_be, y_train, criterion="BIC")


Step:  BIC=4795.064
y ~ const + P_DEMO__RIDAGEYR_Age_in_years_at_screening + P_MCQ__MCQ300C + P_ALB_CR__URDACT_Albumin_creatinine_ratio_mg_g + P_LUX__LUXCAPM_Median_CAP_decibels_per_meter_dB_m + P_BIOPRO__LBXSCLSI_Chloride_mmol_L + Pulse_median + P_BIOPRO__LBDSTRSI_Triglycerides_refrig_serum_mmol_L + P_BMX__BMXWAIST_Waist_Circumference_cm + P_FASTQX__PHDSESN_Session_in_which_SP_was_examined_1.0 + P_TCHOL__LBDTCSI_Total_Cholesterol_mmol_L + P_BPQ__BPQ020_Ever_told_you_had_high_blood_pressure_2.0 + P_PBCD__LBDBPBSI_Blood_lead_umol_L + P_MCQ__MCQ366D + P_BIOPRO__LBXSLDSI_Lactate_Dehydrogenase_LDH_IU_L + P_DEMO__RIDRETH3_Race_Hispanic_origin_w_NH_Asian_3.0 + P_WHQ__WHQ150_Age_when_heaviest_weight + P_PAQ__PAQ620_Moderate_work_activity_2.0 + P_MCQ__MCQ366B + P_LUX__LUXSMED_Median_stiffness_E_kilopascals_kPa + P_TST__LBXSHBG_SHBG_nmol_L + P_WHQ__WHD140_Self_reported_greatest_weight_pounds + P_CBC__LBXMPSI_Mean_platelet_volume_fL + BP_dia_median + P_FASTQX__PHDSESN_Session_in_which_SP_was_ex

In [ ]:
X_train_bic_const = sm.add_constant(X_step_bic_backward.astype(float))
X_test_bic_const  = sm.add_constant(X_test_be[X_step_bic_backward.columns].astype(float))

y_train_prob_bic = model_bic_backward.predict(X_train_bic_const)
y_test_prob_bic  = model_bic_backward.predict(X_test_bic_const)

y_train_pred_bic = (y_train_prob_bic >= 0.5).astype(int)
y_test_pred_bic  = (y_test_prob_bic >= 0.5).astype(int)

acc_train_bic = accuracy_score(y_train_cat, y_train_pred_bic)
acc_test_bic  = accuracy_score(y_test_cat, y_test_pred_bic)
f1_train_bic  = f1_score(y_train_cat, y_train_pred_bic, average='macro')
f1_test_bic   = f1_score(y_test_cat, y_test_pred_bic, average='macro')
auc_train_bic = roc_auc_score(y_train_cat, y_train_prob_bic)
auc_test_bic  = roc_auc_score(y_test_cat, y_test_prob_bic)

print(f"Train ACC: {acc_train_bic:.3f}, F1: {f1_train_bic:.3f}, AUC: {auc_train_bic:.3f}")
print(f"Test  ACC: {acc_test_bic:.3f}, F1: {f1_test_bic:.3f}, AUC: {auc_test_bic:.3f}\n")

print(model_bic_backward.summary())

Train ACC: 0.875, F1: 0.749, AUC: 0.903
Test  ACC: 0.888, F1: 0.764, AUC: 0.900

                 Generalized Linear Model Regression Results                  
Dep. Variable:            IS_DIABETES   No. Observations:                 7789
Model:                            GLM   Df Residuals:                     7747
Model Family:                Binomial   Df Model:                           41
Link Function:                  Logit   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -2183.9
Date:                Tue, 25 Nov 2025   Deviance:                       4367.9
Time:                        11:25:26   Pearson chi2:                 7.12e+03
No. Iterations:                     7   Pseudo R-squ. (CS):             0.3011
Covariance Type:            nonrobust                                         
                                                                coef    std err          z      P>|z|      [0.025      0.975]
---

In [30]:
def stepwise_glm_aic_forward(X, y, criterion="AIC", verbose=True):

    crit = criterion.upper()
    X_current = X.copy()
    iteration = 0
    history = []

    # start with intercept-only model
    X_current = pd.DataFrame(index=X.index)

    base_const = sm.add_constant(X_current.astype(float))
    base_model = sm.GLM(y, base_const, family=sm.families.Binomial()).fit()

    current_model = base_model
    current_rss = current_model.deviance
    current_aic = current_model.aic if crit=="AIC" else current_model.bic

    remaining = list(X.columns)
    iteration = 0

    while True:
        iteration += 1

        if verbose:
            print(f"\nStep:  {crit}={current_aic:.3f}")
            if len(X_current.columns) > 0:
                print("y ~ " + " + ".join(X_current.columns))
            else:
                print("y ~ 1")
            print()
            print(f"{'Df':<6}{'Sum of Sq':<14}{'RSS':<12}{crit:<8}")
            print("-"*50)

        best_score = current_aic
        best_feature = None
        best_model_temp = current_model

        # <none> row first
        if verbose:
            print(f"{'<none>':<6}{'-':<14}{current_rss:<12.3f}{current_aic:<8.3f}")

        for feature in remaining:
            X_temp = pd.concat([X_current, X[[feature]]], axis=1)
            X_temp_const = sm.add_constant(X_temp.astype(float))
            model_temp = sm.GLM(y, X_temp_const, family=sm.families.Binomial()).fit()

            rss = model_temp.deviance
            sumsq = current_rss - rss
            score = model_temp.aic if crit=="AIC" else model_temp.bic

            if verbose:
                print(f"+ {feature:<4}{1:<6}{sumsq:<14.3f}{rss:<12.3f}{score:<8.3f}")

            if score < best_score:
                best_score = score
                best_feature = feature
                best_model_temp = model_temp

        if best_feature is None:
            if verbose:
                print("\nNo improvement — stopping")
            break

        if verbose:
            print(f"\nSelected: + {best_feature}  ({crit}={best_score:.3f})")

        history.append({
            "Step": iteration,
            "Added": best_feature,
            crit: best_score
        })

        # update model
        X_current = pd.concat([X_current, X[[best_feature]]], axis=1)
        remaining.remove(best_feature)
        current_model = best_model_temp
        current_rss = current_model.deviance
        current_aic = best_score

        if len(remaining) == 0:
            if verbose:
                print("\nNo variables left — stopping")
            break

    return X_current, current_model, history


X_step_aic, model_aic, history_aic = stepwise_glm_aic_forward(X_train_be, y_train, criterion="AIC")


Step:  AIC=7160.176
y ~ 1

Df    Sum of Sq     RSS         AIC     
--------------------------------------------------
<none>-             7158.176    7160.176
+ const1     0.000         7158.176    7160.176
+ P_DEMO__RIDAGEYR_Age_in_years_at_screening1     990.705       6167.471    6171.471
+ P_MCQ__MCQ300C1     89.074        7069.102    7073.102
+ P_ALB_CR__URDACT_Albumin_creatinine_ratio_mg_g1     96.478        7061.698    7065.698
+ P_LUX__LUXCAPM_Median_CAP_decibels_per_meter_dB_m1     760.817       6397.359    6401.359
+ P_BIOPRO__LBXSCLSI_Chloride_mmol_L1     162.955       6995.221    6999.221
+ Pulse_median1     36.319        7121.857    7125.857
+ P_BIOPRO__LBDSTRSI_Triglycerides_refrig_serum_mmol_L1     189.801       6968.375    6972.375
+ P_BMX__BMXWAIST_Waist_Circumference_cm1     666.039       6492.137    6496.137
+ P_FASTQX__PHDSESN_Session_in_which_SP_was_examined_1.01     2.468         7155.709    7159.709
+ P_TCHOL__LBDTCSI_Total_Cholesterol_mmol_L1     10.327        

In [48]:
X_train_fwd_const = sm.add_constant(X_step_aic.astype(float))
X_test_fwd_const  = sm.add_constant(X_test_be[X_step_aic.columns].astype(float))

y_train_prob_aic = model_aic.predict(X_train_fwd_const)
y_test_prob_aic  = model_aic.predict(X_test_fwd_const)

y_train_pred_aic = (y_train_prob_aic >= 0.5).astype(int)
y_test_pred_aic  = (y_test_prob_aic >= 0.5).astype(int)

acc_train_aic = accuracy_score(y_train_cat, y_train_pred_aic)
acc_test_aic  = accuracy_score(y_test_cat, y_test_pred_aic)
f1_train_aic  = f1_score(y_train_cat, y_train_pred_aic, average='macro')
f1_test_aic   = f1_score(y_test_cat, y_test_pred_aic, average='macro')
auc_train_aic = roc_auc_score(y_train_cat, y_train_prob_aic)
auc_test_aic  = roc_auc_score(y_test_cat, y_test_prob_aic)

print(f"Train ACC: {acc_train_aic:.3f}, F1: {f1_train_aic:.3f}, AUC: {auc_train_aic:.3f}")
print(f"Test  ACC: {acc_test_aic:.3f}, F1: {f1_test_aic:.3f}, AUC: {auc_test_aic:.3f}\n")

print(model_aic.summary())


Train ACC: 0.875, F1: 0.749, AUC: 0.903
Test  ACC: 0.888, F1: 0.764, AUC: 0.900

                 Generalized Linear Model Regression Results                  
Dep. Variable:            IS_DIABETES   No. Observations:                 7789
Model:                            GLM   Df Residuals:                     7747
Model Family:                Binomial   Df Model:                           41
Link Function:                  Logit   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -2183.9
Date:                Tue, 25 Nov 2025   Deviance:                       4367.9
Time:                        11:26:59   Pearson chi2:                 7.12e+03
No. Iterations:                     7   Pseudo R-squ. (CS):             0.3011
Covariance Type:            nonrobust                                         
                                                                coef    std err          z      P>|z|      [0.025      0.975]
---

In [49]:
X_step_aic, model_aic, history_aic = stepwise_glm_aic_forward(X_train_be, y_train, criterion="BIC")


Step:  BIC=7167.137
y ~ 1

Df    Sum of Sq     RSS         BIC     
--------------------------------------------------
<none>-             7158.176    7167.137
+ const1     0.000         7158.176    7167.137
+ P_DEMO__RIDAGEYR_Age_in_years_at_screening1     990.705       6167.471    6185.392
+ P_MCQ__MCQ300C1     89.074        7069.102    7087.023
+ P_ALB_CR__URDACT_Albumin_creatinine_ratio_mg_g1     96.478        7061.698    7079.619
+ P_LUX__LUXCAPM_Median_CAP_decibels_per_meter_dB_m1     760.817       6397.359    6415.280
+ P_BIOPRO__LBXSCLSI_Chloride_mmol_L1     162.955       6995.221    7013.142
+ Pulse_median1     36.319        7121.857    7139.778
+ P_BIOPRO__LBDSTRSI_Triglycerides_refrig_serum_mmol_L1     189.801       6968.375    6986.296
+ P_BMX__BMXWAIST_Waist_Circumference_cm1     666.039       6492.137    6510.058
+ P_FASTQX__PHDSESN_Session_in_which_SP_was_examined_1.01     2.468         7155.709    7173.630
+ P_TCHOL__LBDTCSI_Total_Cholesterol_mmol_L1     10.327        

In [50]:
X_train_fwd_const = sm.add_constant(X_step_aic.astype(float))
X_test_fwd_const  = sm.add_constant(X_test_be[X_step_aic.columns].astype(float))

y_train_prob_aic = model_aic.predict(X_train_fwd_const)
y_test_prob_aic  = model_aic.predict(X_test_fwd_const)

y_train_pred_aic = (y_train_prob_aic >= 0.5).astype(int)
y_test_pred_aic  = (y_test_prob_aic >= 0.5).astype(int)

acc_train_aic = accuracy_score(y_train_cat, y_train_pred_aic)
acc_test_aic  = accuracy_score(y_test_cat, y_test_pred_aic)
f1_train_aic  = f1_score(y_train_cat, y_train_pred_aic, average='macro')
f1_test_aic   = f1_score(y_test_cat, y_test_pred_aic, average='macro')
auc_train_aic = roc_auc_score(y_train_cat, y_train_prob_aic)
auc_test_aic  = roc_auc_score(y_test_cat, y_test_prob_aic)

print(f"Train ACC: {acc_train_aic:.3f}, F1: {f1_train_aic:.3f}, AUC: {auc_train_aic:.3f}")
print(f"Test  ACC: {acc_test_aic:.3f}, F1: {f1_test_aic:.3f}, AUC: {auc_test_aic:.3f}\n")

print(model_aic.summary())


Train ACC: 0.876, F1: 0.751, AUC: 0.900
Test  ACC: 0.889, F1: 0.763, AUC: 0.896

                 Generalized Linear Model Regression Results                  
Dep. Variable:            IS_DIABETES   No. Observations:                 7789
Model:                            GLM   Df Residuals:                     7751
Model Family:                Binomial   Df Model:                           37
Link Function:                  Logit   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -2207.3
Date:                Tue, 25 Nov 2025   Deviance:                       4414.7
Time:                        11:29:51   Pearson chi2:                 7.31e+03
No. Iterations:                     7   Pseudo R-squ. (CS):             0.2969
Covariance Type:            nonrobust                                         
                                                                coef    std err          z      P>|z|      [0.025      0.975]
---